## Policies Dataset

There are five separate sample policies that we will be using for InsureCheck

The original policy pdfs are stored in the documents folder

The goal here is to create a RAPTOR like index for these policies, so that it can handle specific questions as well as high level questions during the retrieval process

> Original reference for RAPTOR: https://arxiv.org/html/2401.18059v1

### How this works

First, we take the policy pdfs and convert it to txt files for each individual pages in the pdf.

Then we use an LLM to summarize the content for each page and store those summaries as embeddings in our vector store. These are helpful for specific questions related to a page and will be retrieved in those cases.

For more general questions, we create summaries of summaries, i.e. mid level and high level summaries and store these as embeddings as well.

![Policy Indexing](../assets/policy_indexing.png)


Structure of policies dataset:

- documents folder - contains all the original policy documents in pdf format
- pages folder - contains subfolders for each policy which further contains the converted pages for that policy in .txt format
- summary folder - contains the page wise summaries with a subfolder dedicated to each policy
- midsummary and highsummary folders - contains summaries of summaries
- chroma_store - vector store containing the embeddings of all the summaries along with metadata for filtering


### Part 1 - Converting Policy PDF pages to txt files

We use PyPDF2 to read each pdf and extract text from individual pages.

#### Why save as separate text files?

Currently we only have five sample policies and it file based storage is easier to process, debug and make changes

In [ ]:
import os
from PyPDF2 import PdfReader
from tqdm import tqdm # for checking progress

BASE_FOLDER = "."
DOCUMENTS_FOLDER = os.path.join(BASE_FOLDER, "documents")
PAGES_FOLDER = os.path.join(BASE_FOLDER, "pages")
os.makedirs(PAGES_FOLDER, exist_ok=True)

In [7]:
# Test Code

test_file = os.path.join(DOCUMENTS_FOLDER, "three.pdf")
reader = PdfReader(test_file)

# Output content from policy 3 page 34
print(reader.pages[33].extract_text()[:134])

SECTION 1Thingstoknowaboutgettingyourmedical careasa
member ofourplan
Thischapterexplainswhatyouneedtoknowaboutusingtheplantogetyourme


In [8]:
# Main Code for conversion of all policy pdf pages to txt files

for pdf_filename in tqdm(os.listdir(DOCUMENTS_FOLDER), desc="Converting PDFs to text"):
    if pdf_filename.endswith(".pdf"):
        
        # Get policy name and make folder for policy in the pages/ folder
        policy_name = pdf_filename.replace(".pdf", "")
        policy_output_folder = os.path.join(PAGES_FOLDER, policy_name)
        os.makedirs(policy_output_folder, exist_ok=True)
        
        pdf_path = os.path.join(DOCUMENTS_FOLDER, pdf_filename)
        reader = PdfReader(pdf_path)
        number_of_pages = len(reader.pages)
        
        print("\nNow converting", pdf_filename,"which has", number_of_pages,"pages")
        
        for page_number in range(number_of_pages):

            current_page = reader.pages[page_number]
            extracted_text = current_page.extract_text()
            
            if not extracted_text or extracted_text.strip() == "":
                extracted_text = f"[No extractable text found on page {page_number + 1}]"
                print(extracted_text)
            
            output_filename = f"page{page_number + 1}.txt"
            output_filepath = os.path.join(policy_output_folder, output_filename)
            
            with open(output_filepath, "w", encoding="utf-8") as text_file:
                text_file.write(extracted_text)
        
        print("Finished converting", pdf_filename)


Converting PDFs to text:   0%|          | 0/5 [00:00<?, ?it/s]


Now converting five.pdf which has 116 pages


Converting PDFs to text:  20%|██        | 1/5 [00:03<00:12,  3.14s/it]

Finished converting five.pdf

Now converting four.pdf which has 174 pages


Converting PDFs to text:  40%|████      | 2/5 [00:07<00:10,  3.64s/it]

...

Finished converting one.pdf

Now converting three.pdf which has 188 pages
[No extractable text found on page 2]


Converting PDFs to text:  80%|████████  | 4/5 [00:14<00:03,  3.78s/it]

[No extractable text found on page 185]
[No extractable text found on page 186]
[No extractable text found on page 187]
Finished converting three.pdf

Now converting two.pdf which has 89 pages


Converting PDFs to text: 100%|██████████| 5/5 [00:17<00:00,  3.53s/it]

Finished converting two.pdf


### Part 2 - Creating Summaries of Each Page

We now create summaries for each policy page.

#### Why summarize?

- Makes the extracted data cleaner and more concise
- The embeddings model all-MiniLM-L6-v2 that we will be using has a token limit of 256 word pieces after which the data is truncated. Thus creating summaries helps avoid data loss

> https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2

#### How it works

We use a method similar to interleaving where each summary page receives the context of the previous page's summary, which helps the LLM understand references to the earlier content and creating coherent summaries.

![Page Wise Summary Creation](../assets/policy_page_summary.png)

> Original Interleaving Idea Reference: https://arxiv.org/pdf/2212.10509

In [2]:
import os
from transformers import AutoTokenizer
from langchain_community.llms import Ollama
from tqdm import tqdm

llm = Ollama(model="gemma3:1b")

BASE_FOLDER = "."
PAGES_FOLDER = os.path.join(BASE_FOLDER, "pages")
SUMMARY_FOLDER = os.path.join(BASE_FOLDER, "summary")
os.makedirs(SUMMARY_FOLDER, exist_ok=True)

# each summary needs to be within a token limit hence initilaizing a tokenizer for token counting
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
def count_tokens(text):
    tokens = tokenizer.encode(text)
    return len(tokens)

In [ ]:
# Test Code
data = reader.pages[33].extract_text()

prompt = f"""

VERY IMPORTANT: Give only the summary directly as the final response. Nothing before of after it and no other explanations. JUST the final summary.

You are an expert insurance policy analyst. Your task is to create a concise, information-dense summary of the current page
Create a summary of the following in less than 180 words

{data}

"""
llm.invoke(prompt)

'This chapter outlines how to utilize the plan for medical services. It defines network providers – healthcare professionals and facilities that have agreements to provide covered services. You’ll pay only your share of the cost for these providers’ services directly. Covered services include medical care, health care services, supplies, equipment, and prescription drugs.  The benefits chart in Chapter 4 details what services are covered and how much you pay.  As a Medicare health plan member, you can receive care from either a network provider or an out-of-network provider.'

In [5]:
def create_initial_summary_prompt(current_page_content, previous_summary=None):
    if previous_summary:
        prompt = f"""

VERY IMPORTANT: Give only the summary directly as the final response. Nothing before of after it and no other explanations. JUST the final summary.

You are an expert insurance policy analyst. Your task is to create a concise, information-dense summary of the current page while maintaining continuity with previous content.

PREVIOUS PAGE CONTEXT:
{previous_summary}

CURRENT PAGE CONTENT:
{current_page_content}

Create a summary in LESS than 180 words that:
1. Preserves all critical insurance terms, coverage amounts, conditions, and exclusions
2. Maintains key policy definitions and clauses
3. References previous context only if directly relevant to this page
4. Uses precise insurance terminology
5. Structures information clearly with important details prioritized

Focus on what's NEW or IMPORTANT on this page. Be concise but complete.

SUMMARY:

"""
        
    else:
        prompt = f"""

VERY IMPORTANT: Give only the summary directly as the final response. Nothing before of after it and no other explanations. JUST the final summary.
        
You are an expert insurance policy analyst. Your task is to create a concise, information-dense summary of the first page of this insurance policy.

CURRENT PAGE CONTENT:
{current_page_content}

Create a summary in LESS than 180 words that:
1. Preserves all critical insurance terms, coverage amounts, conditions, and exclusions
2. Maintains key policy definitions and clauses
3. Uses precise insurance terminology
4. Structures information clearly with important details prioritized
5. Captures the essential information from this page

Be concise but complete.

SUMMARY:

"""
    
    return prompt

In [6]:
def create_compression_prompt(summary, current_token_count, target_tokens=240):
    prompt = f"""
    
The following insurance policy summary is too long ({current_token_count} tokens). 
Compress it to under {target_tokens} tokens while preserving ALL critical information:
- Coverage amounts and limits
- Key terms and definitions
- Important conditions and exclusions
- Policy numbers and dates

CURRENT SUMMARY:
{summary}

VERY IMPORTANT: Provide ONLY the compressed summary with no explanations. Maintain technical accuracy.

COMPRESSED SUMMARY:

"""
    
    return prompt

In [8]:
def generate_summary_with_token_limit(page_content, previous_summary=None, max_tokens=256, max_attempts=5):

    # page_content - content of the current page
    # max_tokens - maximum allowed tokens for embedding purposes
    # max_attempts - maximum compression attempts
    
    initial_prompt = create_initial_summary_prompt(page_content, previous_summary)
    summary = llm.invoke(initial_prompt).strip()
    token_count = count_tokens(summary)
    if token_count <= max_tokens:
        return summary, token_count, True
    
    # If summary exceeds token limit, attempt compression

    attempt = 1
    target_tokens = max_tokens - 10
    
    while token_count > max_tokens and attempt <= max_attempts:
        # print(f"Compression attempt {attempt}")
        
        compression_prompt = create_compression_prompt(summary, token_count, target_tokens)
        summary = llm.invoke(compression_prompt).strip()
        token_count = count_tokens(summary)       
        attempt += 1
    
    success = token_count <= max_tokens

    if not success:
        print("Compression Failed! Summary Exceeds token count")
    
    return summary, token_count, success

In [9]:
# Main code for creating summaries for all policies

print("Starting policy pages summarization")
policy_folders = [f for f in os.listdir(PAGES_FOLDER) if os.path.isdir(os.path.join(PAGES_FOLDER, f))]

for policy_name in tqdm(policy_folders, desc="Processing policies", position = 0):

    print("\n\nProcessing Policy:", policy_name)
    policy_pages_folder = os.path.join(PAGES_FOLDER, policy_name)
    policy_summary_folder = os.path.join(SUMMARY_FOLDER, policy_name)
    os.makedirs(policy_summary_folder, exist_ok=True)
    
    # Sorting pages numerically
    page_files = [f for f in os.listdir(policy_pages_folder) if f.endswith('.txt')]
    page_files.sort(key=lambda x: int(x.replace('page', '').replace('.txt', '')))
    
    previous_summary = None
    
    for page_file in tqdm(page_files, desc=f"Pages in {policy_name}", leave=False, position = 1):
        page_number = page_file.replace('page', '').replace('.txt', '')
        # print(f"Page {page_number}. ", end=" ")
        
        page_path = os.path.join(policy_pages_folder, page_file)
        with open(page_path, 'r', encoding='utf-8') as f:
            page_content = f.read()

        if not page_content.strip() or "[No extractable text found" in page_content:
            print(f"Skipping empty page")
            summary_text = "[Empty or unreadable page - no summary generated]"
            previous_summary = None
        else:
            # Generate summary
            summary_text, token_count, success = generate_summary_with_token_limit(
                page_content, 
                previous_summary,
                max_tokens=256
            )
            previous_summary = summary_text
            
            if success:
                pass
            else:
                print(f"Summary exceeds limit: {token_count} tokens")
        
        summary_filename = f"page{page_number}summary.txt"
        summary_path = os.path.join(policy_summary_folder, summary_filename)
        
        with open(summary_path, 'w', encoding='utf-8') as f:
            f.write(summary_text)


print("\n\nSummarization complete!")

Starting policy pages summarization


Processing policies:   0%|          | 0/5 [00:00<?, ?it/s]



Processing Policy: five


Compression Failed! Summary Exceeds token count
Summary exceeds limit: 272 tokens


Compression Failed! Summary Exceeds token count
Summary exceeds limit: 258 tokens


...

Processing policies:  80%|████████  | 4/5 [4:25:11<1:08:25, 4105.23s/it]



Processing Policy: two


Compression Failed! Summary Exceeds token count
Summary exceeds limit: 279 tokens


Processing policies: 100%|██████████| 5/5 [5:05:23<00:00, 3664.75s/it]  



Summarization complete!


### Part 3 - Generating mid and high level summaries

We now create mid-level and high-level summaries by grouping lower-level summaries together.
- Mid Level Summaries - Group 5 consecutive page summaries into one summary covering ~5 pages
- High Level Summaries - Group 5 consecutive mid summaries into one summary covering ~25 pages

#### Why hierarchical summaries?

- Enables answering both specific questions (page-level) and broad questions (high-level)
- Reduces total number of embeddings while maintaining information
- Implements RAPTOR-like hierarchical retrieval for better search performance

> RAPTOR Reference: https://arxiv.org/html/2401.18059v1

In [22]:
import os
from transformers import AutoTokenizer
from langchain_community.llms import Ollama
from tqdm import tqdm

llm = Ollama(model="gemma3:1b")

BASE_FOLDER = "."
SUMMARY_FOLDER = os.path.join(BASE_FOLDER, "summary")
MIDSUMMARY_FOLDER = os.path.join(BASE_FOLDER, "midsummary")
HIGHSUMMARY_FOLDER = os.path.join(BASE_FOLDER, "highsummary")
os.makedirs(MIDSUMMARY_FOLDER, exist_ok=True)
os.makedirs(HIGHSUMMARY_FOLDER, exist_ok=True)

tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
def count_tokens(text):
    tokens = tokenizer.encode(text)
    return len(tokens)

def get_page_number(filename):
    return int(filename.replace('page', '').replace('summary.txt', ''))

In [25]:
def create_group_summary_prompt(summaries_list, page_range, level="mid"):
    
    combined_summaries = "\n\n---PAGE BREAK---\n\n".join(summaries_list)
    
    if level == "mid":
        prompt = f"""

VERY IMPORTANT: Give only the summary directly as the final response. Nothing before or after it and no other explanations. JUST the final summary.

You are an expert insurance policy analyst. You are given summaries of pages {page_range} from an insurance policy document.

Your task is to create a UNIFIED mid-level summary that synthesizes these page summaries into a coherent overview.

INDIVIDUAL PAGE SUMMARIES:
{combined_summaries}

Create a consolidated summary in LESS than 180 words that:
1. Identifies and preserves ALL critical insurance information across these pages:
   - Coverage amounts, limits, and conditions
   - Key exclusions and restrictions
   - Important definitions and terminology
   - Policy numbers, dates, and contact information
2. Groups related information together (e.g., all coverage details, all exclusions)
3. Eliminates redundancy while maintaining completeness
4. Maintains the logical flow of information across pages {page_range}
5. Uses precise insurance terminology

Focus on creating a cohesive narrative that captures the essential information from pages {page_range}.

MID-LEVEL SUMMARY:
"""
    
    # high level
    else:
        prompt = f"""
        
VERY IMPORTANT: Give only the summary directly as the final response. Nothing before or after it and no other explanations. JUST the final summary.

You are an expert insurance policy analyst. You are given mid-level summaries covering pages {page_range} from an insurance policy document.

Your task is to create a HIGH-LEVEL summary that provides a comprehensive overview of this major section.

MID-LEVEL SUMMARIES:
{combined_summaries}

Create a high-level consolidated summary in LESS than 180 words that:
1. Captures the overarching themes and major provisions across pages {page_range}
2. Preserves ONLY the most critical information:
   - Major coverage categories and their limits
   - Key exclusions and conditions
   - Important policy-level definitions
   - Essential contact information or policy identifiers
3. Provides a bird's-eye view suitable for understanding the general scope
4. Maintains logical structure and flow
5. Uses clear, precise insurance terminology

This summary should enable someone to understand the main content and structure of pages {page_range}.

HIGH-LEVEL SUMMARY:
"""
    
    return prompt

In [26]:
def create_group_compression_prompt(summary, current_token_count, target_tokens=240, level="mid"):
    
    level_text = "mid-level" if level == "mid" else "high-level"
    
    prompt = f"""

VERY IMPORTANT: Provide ONLY the compressed summary with no explanations. Maintain technical accuracy.

The following {level_text} insurance policy summary is too long ({current_token_count} tokens).
Compress it to under {target_tokens} tokens while preserving ALL critical information:
- Coverage amounts and limits
- Key terms and definitions
- Important conditions and exclusions
- Policy numbers and dates
- Major themes and provisions

CURRENT SUMMARY:
{summary}

COMPRESSED SUMMARY:
"""
    
    return prompt

In [27]:
def generate_grouped_summary(summaries_list, page_range, level="mid", max_tokens=256, max_attempts=5):
    
    initial_prompt = create_group_summary_prompt(summaries_list, page_range, level)
    summary = llm.invoke(initial_prompt).strip()
    token_count = count_tokens(summary)
    if token_count <= max_tokens:
        return summary, token_count, True
    
    # Compression
    attempt = 1
    target_tokens = max_tokens - 10
    
    while token_count > max_tokens and attempt <= max_attempts:
        compression_prompt = create_group_compression_prompt(summary, token_count, target_tokens, level)
        summary = llm.invoke(compression_prompt).strip()
        token_count = count_tokens(summary)
        attempt += 1
    
    success = token_count <= max_tokens
    
    if not success:
        tqdm.write(f"Compression Failed! Summary exceeds {max_tokens} tokens: {token_count}")
    
    return summary, token_count, success

In [28]:
print("Starting mid-level summarization")

policy_folders = [f for f in os.listdir(SUMMARY_FOLDER) if os.path.isdir(os.path.join(SUMMARY_FOLDER, f))]

for policy_name in tqdm(policy_folders, desc="Processing policies for mid-summaries", position=0):
    print(f"\n\nProcessing Policy: {policy_name}")
    
    policy_summary_folder = os.path.join(SUMMARY_FOLDER, policy_name)
    policy_midsummary_folder = os.path.join(MIDSUMMARY_FOLDER, policy_name)
    os.makedirs(policy_midsummary_folder, exist_ok=True)
    
    # Sorting
    summary_files = [f for f in os.listdir(policy_summary_folder) if f.endswith('summary.txt')]
    summary_files.sort(key=get_page_number)
    
    group_size = 5
    total_files = len(summary_files)
    
    for i in tqdm(range(0, total_files, group_size), desc=f"Creating mid-summaries", leave=False, position=1):
        
        batch = summary_files[i:i + group_size]
        start_page = get_page_number(batch[0])
        end_page = get_page_number(batch[-1])
        
        if start_page == end_page:
            page_range = f"{start_page}"
            output_filename = f"page{start_page}to{end_page}.txt"
        else:
            page_range = f"{start_page}-{end_page}"
            output_filename = f"page{start_page}to{end_page}.txt"
        
        summaries_list = []
        for summary_file in batch:
            summary_path = os.path.join(policy_summary_folder, summary_file)
            with open(summary_path, 'r', encoding='utf-8') as f:
                content = f.read()
                if "[Empty or unreadable page" not in content:
                    summaries_list.append(content)
        
        if not summaries_list:
            mid_summary = "[All pages in this range were empty or unreadable]"
            token_count = 0
        else:
            mid_summary, token_count, success = generate_grouped_summary(
                summaries_list,
                page_range,
                level="mid",
                max_tokens=256
            )

        output_path = os.path.join(policy_midsummary_folder, output_filename)
        with open(output_path, 'w', encoding='utf-8') as f:
            f.write(mid_summary)

print("\n\nMid-level summarization complete!")

Starting mid-level summarization


Processing policies for mid-summaries:   0%|          | 0/5 [00:00<?, ?it/s]



Processing Policy: five


Processing policies for mid-summaries:  20%|██        | 1/5 [12:12<48:49, 732.48s/it]



Processing Policy: four


...



Processing Policy: three


Processing policies for mid-summaries:  80%|████████  | 4/5 [56:49<14:37, 877.47s/it]



Processing Policy: two


Processing policies for mid-summaries: 100%|██████████| 5/5 [1:06:04<00:00, 792.95s/it]



Mid-level summarization complete!


In [29]:
print("\n\nStarting high-level summarization")

policy_folders = [f for f in os.listdir(MIDSUMMARY_FOLDER) if os.path.isdir(os.path.join(MIDSUMMARY_FOLDER, f))]

for policy_name in tqdm(policy_folders, desc="Processing policies for high-summaries", position=0):
    print(f"\n\nProcessing Policy: {policy_name}")
    
    policy_midsummary_folder = os.path.join(MIDSUMMARY_FOLDER, policy_name)
    policy_highsummary_folder = os.path.join(HIGHSUMMARY_FOLDER, policy_name)
    os.makedirs(policy_highsummary_folder, exist_ok=True)
    
    midsummary_files = [f for f in os.listdir(policy_midsummary_folder) if f.endswith('.txt')]
    
    def get_start_page(filename):
        return int(filename.replace('page', '').split('to')[0])
    
    midsummary_files.sort(key=get_start_page)
    
    group_size = 5
    total_files = len(midsummary_files)
    
    for i in tqdm(range(0, total_files, group_size), desc=f"Creating high-summaries", leave=False, position=1):

        batch = midsummary_files[i:i + group_size]
        
        start_page = get_start_page(batch[0])
        end_page = int(batch[-1].replace('page', '').split('to')[1].replace('.txt', ''))
        
        if start_page == end_page:
            page_range = f"{start_page}"
            output_filename = f"page{start_page}to{end_page}.txt"
        else:
            page_range = f"{start_page}-{end_page}"
            output_filename = f"page{start_page}to{end_page}.txt"
        
        midsummaries_list = []
        for midsummary_file in batch:
            midsummary_path = os.path.join(policy_midsummary_folder, midsummary_file)
            with open(midsummary_path, 'r', encoding='utf-8') as f:
                content = f.read()
                if "[All pages in this range were empty" not in content:
                    midsummaries_list.append(content)
        
        if not midsummaries_list:
            high_summary = "[All pages in this range were empty or unreadable]"
            token_count = 0
        else:
            high_summary, token_count, success = generate_grouped_summary(
                midsummaries_list,
                page_range,
                level="high",
                max_tokens=256
            )

        output_path = os.path.join(policy_highsummary_folder, output_filename)
        with open(output_path, 'w', encoding='utf-8') as f:
            f.write(high_summary)

print("\n\nHigh-level summarization complete!")



Starting high-level summarization


Processing policies for high-summaries:   0%|          | 0/5 [00:00<?, ?it/s]



Processing Policy: five


Processing policies for high-summaries:  20%|██        | 1/5 [01:58<07:52, 118.01s/it]



Processing Policy: four


...



Processing Policy: three


Processing policies for high-summaries:  80%|████████  | 4/5 [10:25<02:45, 165.27s/it]



Processing Policy: two


Processing policies for high-summaries: 100%|██████████| 5/5 [11:57<00:00, 143.56s/it]



High-level summarization complete!


In [30]:
print("ALL SUMMARIZATIONS COMPLETE!")

for policy_name in policy_folders:
    page_summaries = len([f for f in os.listdir(os.path.join(SUMMARY_FOLDER, policy_name)) if f.endswith('.txt')])
    mid_summaries = len([f for f in os.listdir(os.path.join(MIDSUMMARY_FOLDER, policy_name)) if f.endswith('.txt')])
    high_summaries = len([f for f in os.listdir(os.path.join(HIGHSUMMARY_FOLDER, policy_name)) if f.endswith('.txt')])
    
    print(f"\nPolicy: {policy_name}")
    print(f"Page summaries: {page_summaries}")
    print(f"Mid summaries: {mid_summaries}")
    print(f"High summaries: {high_summaries}")

ALL SUMMARIZATIONS COMPLETE!

Policy: five
Page summaries: 116
Mid summaries: 24
High summaries: 5

Policy: four
Page summaries: 174
Mid summaries: 35
High summaries: 7

Policy: one
Page summaries: 135
Mid summaries: 27
High summaries: 6

Policy: three
Page summaries: 188
Mid summaries: 38
High summaries: 8

Policy: two
Page summaries: 89
Mid summaries: 18
High summaries: 4


### Part 4 - Storing Summary Embeddings in a Vector Store

We now convert all summaries (page, mid, and high-level) into embeddings and store them in a vector database.

By storing in a vector database we can use semantic search later and we get the option to use metadata filters which will allow us to search for specific policies, page ranges or summary levels

Each summary is converted to a 384-dimensional vector using the all-MiniLM-L6-v2 embedding model. These vectors are stored in ChromaDB along with metadata policy name, summary type, page range and filename.

```persist_directory``` tells Chroma to save it to the disk locally

We also convert the summaries to document objects first to store them as embeddings in Chroma DB

In [3]:
import os
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.documents import Document
from tqdm import tqdm

BASE_FOLDER = "."
SUMMARY_FOLDER = os.path.join(BASE_FOLDER, "summary")
MIDSUMMARY_FOLDER = os.path.join(BASE_FOLDER, "midsummary")
HIGHSUMMARY_FOLDER = os.path.join(BASE_FOLDER, "highsummary")
CHROMA_STORE_PATH = os.path.join(BASE_FOLDER, "chroma_store")

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

def parse_page_range(filename):
    name = filename.replace('.txt', '')
    if 'summary' in name:
        page_num = int(name.replace('page', '').replace('summary', ''))
        return (page_num, page_num)
    else:
        parts = name.replace('page', '').split('to')
        return (int(parts[0]), int(parts[1]))

In [4]:
def create_documents_from_folder(folder_path, policy_name, summary_type):

    documents = []
    
    summary_files = [f for f in os.listdir(folder_path) if f.endswith('.txt')]
    
    for summary_file in summary_files:
        file_path = os.path.join(folder_path, summary_file)
        
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
        
        if "[Empty or unreadable page" in content or "[All pages in this range were empty" in content:
            continue

        start_page, end_page = parse_page_range(summary_file)

        metadata = {
            "policy_name": policy_name,
            "summary_type": summary_type,
            "start_page": start_page,
            "end_page": end_page,
            "page_range": f"{start_page}-{end_page}" if start_page != end_page else str(start_page),
            "source_file": summary_file
        }

        doc = Document(
            page_content=content,
            metadata=metadata
        )
        
        documents.append(doc)
    
    return documents

In [5]:
print("Starting Embedding Processss")

all_documents = []

policy_folders = [f for f in os.listdir(SUMMARY_FOLDER) if os.path.isdir(os.path.join(SUMMARY_FOLDER, f))]

print(f"\nFound {len(policy_folders)} policies to process")

for policy_name in tqdm(policy_folders, desc="Collecting documents", position=0):

    page_summary_folder = os.path.join(SUMMARY_FOLDER, policy_name)
    page_docs = create_documents_from_folder(page_summary_folder, policy_name, "page")
    all_documents.extend(page_docs)
    
    mid_summary_folder = os.path.join(MIDSUMMARY_FOLDER, policy_name)
    if os.path.exists(mid_summary_folder):
        mid_docs = create_documents_from_folder(mid_summary_folder, policy_name, "mid")
        all_documents.extend(mid_docs)

    high_summary_folder = os.path.join(HIGHSUMMARY_FOLDER, policy_name)
    if os.path.exists(high_summary_folder):
        high_docs = create_documents_from_folder(high_summary_folder, policy_name, "high")
        all_documents.extend(high_docs)

print(f"\nTotal documents collected: {len(all_documents)}")

page_count = sum(1 for doc in all_documents if doc.metadata['summary_type'] == 'page')
mid_count = sum(1 for doc in all_documents if doc.metadata['summary_type'] == 'mid')
high_count = sum(1 for doc in all_documents if doc.metadata['summary_type'] == 'high')

print(f"    Page-level summaries: {page_count}")
print(f"    Mid-level summaries: {mid_count}")
print(f"    High-level summaries: {high_count}")

# Create ChromaDB vector store with all documents
print("Creating embeddings and storing in ChromaDB")

vectorstore = Chroma.from_documents(
    documents=all_documents,
    embedding=embedding_model,
    persist_directory=CHROMA_STORE_PATH
)

print("Process Completed!")
print(f"Vector store saved to: {CHROMA_STORE_PATH}")
print(f"Total embeddings created: {len(all_documents)}")

Starting Embedding Processss

Found 5 policies to process



Total documents collected: 869
    Page-level summaries: 697
    Mid-level summaries: 142
    High-level summaries: 30
Creating embeddings and storing in ChromaDB
Process Completed!
Vector store saved to: .\chroma_store
Total embeddings created: 869


### Examples of Retrieval of Documents from Vector Store

We can now use the vector store and test different queries on policies to see the working retrieval process

In [7]:
import os
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

def search_policies(query, policy_names=None, k=5, chroma_store_path="./chroma_store"):    

    embedding_model = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )
    
    vectorstore = Chroma(
        persist_directory=chroma_store_path,
        embedding_function=embedding_model
    )

    if policy_names is None:
        return vectorstore.similarity_search(query, k=k)

    return vectorstore.similarity_search(
        query, 
        k=k, 
        filter={"policy_name": {"$in": policy_names}}
    )

In [9]:
question = "What is the claim process?"
results = search_policies(
    query = question,
    policy_names = ['one', 'two'],
    k = 3
)

print("Found", len(results), "results")
for r in results:
    print(f"Policy: {r.metadata['policy_name']}", end = " | ")
    print(f"Type: {r.metadata['summary_type']}", end = " | ")
    print(f"Pages: {r.metadata['page_range']}")
    print(f"Content: {r.page_content[:134]}")
    print("-"*34)

# For LLM use
from langchain_community.llms import Ollama
llm = Ollama(model="gemma3:1b")

context = "\n\n".join([r.page_content for r in results])

prompt = f"""

Based on this context:

{context}

Answer the following question:

{question}

Answer:

"""

print(llm.invoke(prompt))

Found 3 results
Policy: two | Type: page | Pages: 76
Content: **Summary:**

This policy governs First Choice Next’s reimbursement of out-of-network provider claims, governed by strict requirements
----------------------------------
Policy: two | Type: page | Pages: 78
Content: **Summary**

This First Choice Next reimbursement policy establishes stringent requirements for out-of-network provider claims. Mandat
----------------------------------
Policy: two | Type: page | Pages: 77
Content: **Summary:**

First Choice Next’s reimbursement policy governs out-of-network provider claims, governed by strict requirements.  This 
----------------------------------
The claim process involves the following steps:

1.  **Prior Authorization:** Mandatory for most services.
2.  **Submission of Claim Forms:** Complete claim forms, proof of loss, and a written statement of loss are required.
3.  **Deadline for Invalidation:** If not met within five days, the claim is invalid.
4.  **Expedited Reviews:*